# SIH26142 — Deep Super-Resolution Fine-Tuning (Google Colab)
### Sentinel-2 ×4 Super-Resolution | Real-ESRGAN (RRDBNet Backbone)

**What this notebook does:**
1. Checks GPU, installs dependencies
2. Clones the project from GitHub
3. Verifies / generates synthetic training pairs
4. Runs multi-loss fine-tuning (L1 + Perceptual + SAM + Edge + FFT)
5. Plots training curves
6. Downloads the best checkpoint back to this machine

> **Recommended runtime:** Runtime → Change runtime type → **T4 GPU**

## Cell 1 — GPU Check
Must show a CUDA GPU. If not, change runtime to T4 GPU.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f'VRAM: {props.total_memory / 1024**3:.1f} GB')
else:
    raise RuntimeError('No GPU detected. Go to Runtime > Change runtime type and select T4 GPU.')


## Cell 2 — Install Dependencies
Installs rasterio, scikit-image, realesrgan and supporting packages.

In [ ]:
# 1. Install standard scientific & image packages first
!pip install -q rasterio scikit-image tqdm opencv-python matplotlib

# 2. Create physical torchvision shim for functional_tensor
import torchvision, os
tv_dir = os.path.dirname(torchvision.__file__)
ft_path = os.path.join(tv_dir, 'transforms', 'functional_tensor.py')
with open(ft_path, 'w') as f:
    f.write('from torchvision.transforms.functional import *\n')

# 3. Install basicsr (optional, fallback provided) & realesrgan
!pip install -q --no-build-isolation basicsr || true
!pip install -q realesrgan || true
print('Dependencies ready.')


## Cell 3 — Clone Repository
Clones your GitHub repo. **Edit the URL below** if your repo path differs.

In [ ]:
import os, sys

REPO_URL = 'https://github.com/AjayBora002/Depth-Wizard.git'  # <-- your repo
CLONE_DIR = '/content/Depth-Wizard'
PROJECT_DIR = f'{CLONE_DIR}/srm-project'

if not os.path.exists(CLONE_DIR):
    !git clone {REPO_URL} {CLONE_DIR}
else:
    print('Repo already cloned, pulling latest...')
    !git -C {CLONE_DIR} pull

# Clear previously imported src modules so new git pull changes take effect
for mod in [m for m in list(sys.modules.keys()) if m.startswith('src')]:
    del sys.modules[mod]

# Add project root to path so 'src' is importable
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

os.chdir(PROJECT_DIR)
print('Working directory:', os.getcwd())
!ls src/


## Cell 4 — Download Pretrained Weights
Auto-downloads `RealESRGAN_x4plus.pth` into `src/checkpoints/` if not already present.

In [ ]:
from src.model import _download_weights
weight_path = _download_weights('x4plus')
print('Weights ready at:', weight_path)


## Cell 5 — Generate Synthetic Training Pairs
If pairs already exist in `data/synthetic_pairs/`, this step is skipped.
Otherwise it synthesises LR/HR pairs from the Sentinel-2 tiles in `data/raw/`.

> **Note:** The repo should already contain sample `.npy` pairs committed to `data/synthetic_pairs/`.
> If not, you need to either upload tiles to `data/raw/` and run generation,
> or upload the pairs directly.

In [ ]:
import os
from pathlib import Path

lr_dir = Path('data/synthetic_pairs/lr')
hr_dir = Path('data/synthetic_pairs/hr')
n_lr = len(list(lr_dir.glob('*.npy'))) if lr_dir.exists() else 0
n_hr = len(list(hr_dir.glob('*.npy'))) if hr_dir.exists() else 0
print(f'Found {n_lr} LR pairs and {n_hr} HR pairs')

if n_lr == 0 or n_hr == 0:
    raw_tiles = list(Path('data/raw').glob('*.tif'))
    if not raw_tiles:
        print('WARNING: No tiles in data/raw/ and no pairs found.')
        print('Upload .tif tiles to data/raw/ via Files panel, then re-run this cell.')
    else:
        print(f'Generating pairs from {len(raw_tiles)} tile(s)...')
        from src.pair_generation import generate_all_pairs
        generate_all_pairs(
            raw_dir='data/raw',
            out_dir='data/synthetic_pairs',
            scale=4,
            patches_per_tile=50,
        )
        print('Pair generation complete.')
else:
    print(f'Pairs already present ({n_lr} LR / {n_hr} HR). Skipping generation.')


## Cell 6 — Fine-Tune the Model
Runs multi-loss training: **L1 + Perceptual (VGG16) + SAM + Edge + FFT**.

Typical Colab T4 speed: **~1–2 min/epoch** for `crop_size=128, batch_size=4`.
50 epochs ≈ 50–100 minutes total.

The `time_budget_hours=2` cap is a safety net — adjust or remove as needed.

In [ ]:
# Clear cached src modules so updated files from git pull are reloaded
import sys
for mod in [m for m in list(sys.modules.keys()) if m.startswith('src')]:
    del sys.modules[mod]

from src.train import train

results = train(
    pairs_dir='data/synthetic_pairs',
    output_dir='src/checkpoints',
    model_key='x4plus',
    epochs=50,
    batch_size=4,
    crop_size=128,
    lr=1e-4,
    lambda_perceptual=0.1,
    lambda_sam=0.05,
    lambda_edge=0.05,
    lambda_freq=0.02,
    save_every=10,
    num_workers=2,
    use_amp=True,           # AMP (FP16) for T4 speed boost
    time_budget_hours=2.0, # Stop cleanly if Colab session nears timeout
)

print('\nTraining complete!')
print('Best checkpoint:', results['best_checkpoint'])
hist = results['history']
print(f'Epochs run: {len(hist["train_loss"])}')
print(f'Best Val PSNR: {max(hist["val_psnr"]):.2f} dB')
print(f'Total training time: {sum(hist["epoch_time"])/60:.1f} min')


## Cell 7 — Training Curves

In [ ]:
import json
import matplotlib.pyplot as plt

with open('src/checkpoints/training_history.json') as f:
    hist = json.load(f)

epochs = range(1, len(hist['train_loss']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs, hist['train_loss'], 'b-o', markersize=4, label='Train')
axes[0].plot(epochs, hist['val_loss'], 'r-o', markersize=4, label='Val')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, hist['val_psnr'], 'g-o', markersize=4)
axes[1].set_title('Validation PSNR (dB)')
axes[1].set_xlabel('Epoch')
axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs, hist['epoch_time'], 'm-o', markersize=4)
axes[2].set_title('Epoch Time (s)')
axes[2].set_xlabel('Epoch')
axes[2].grid(True, alpha=0.3)

plt.suptitle('SIH26142 — SR Fine-Tuning Progress', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('src/checkpoints/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Curves saved to src/checkpoints/training_curves.png')


## Cell 8 — Run 4× Super-Resolution Inference (GPU Accelerated)
Runs full pipeline inference on Sentinel-2 tiles with **FP16 (`--half`)**, **Hann-window blending**, and **fast batched TTA uncertainty quantification**.

On a T4 GPU, a 512×512 tile processes in **~3–5 seconds**!

In [ ]:
!python src/inference.py \
    --input data/raw/S2_Delhi_Sample_512.tif \
    --output data/outputs/sr_delhi_512_finetuned.tif \
    --checkpoint src/checkpoints/model_finetuned_best.pth \
    --half \
    --uncertainty \
    --tta-n 4 \
    --tile-size 256


## Cell 9 — High-Resolution Visualizer & Uncertainty Map
Displays side-by-side comparison: Input Tile vs 4× Super-Resolved vs TTA Uncertainty.

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import os

def norm_rgb(arr):
    rgb = arr[:3].transpose(1, 2, 0).astype(np.float32)
    p2, p98 = np.percentile(rgb, (2, 98))
    if p98 > p2:
        rgb = np.clip((rgb - p2) / (p98 - p2), 0, 1)
    else:
        rgb = np.clip(rgb / 255.0, 0, 1)
    return rgb

with rasterio.open('data/raw/S2_Delhi_Sample_512.tif') as f_lr:
    lr_data = f_lr.read()
    lr_rgb = norm_rgb(lr_data)

with rasterio.open('data/outputs/sr_delhi_512_finetuned.tif') as f_sr:
    sr_data = f_sr.read()
    sr_rgb = norm_rgb(sr_data)

unc_path = 'data/outputs/sr_delhi_512_finetuned_uncertainty.npy'
has_unc = os.path.exists(unc_path)
if has_unc:
    unc = np.load(unc_path)
    unc_map = np.mean(unc, axis=0) if unc.ndim == 3 else unc

fig, axes = plt.subplots(1, 3 if has_unc else 2, figsize=(18, 6))
axes[0].imshow(lr_rgb)
axes[0].set_title(f'Low-Res Input ({lr_rgb.shape[1]}x{lr_rgb.shape[0]})', fontsize=12)
axes[0].axis('off')

axes[1].imshow(sr_rgb)
axes[1].set_title(f'4x Super-Resolved Output ({sr_rgb.shape[1]}x{sr_rgb.shape[0]})', fontsize=12, fontweight='bold', color='green')
axes[1].axis('off')

if has_unc:
    im = axes[2].imshow(unc_map, cmap='viridis')
    axes[2].set_title('TTA Uncertainty Map', fontsize=12)
    axes[2].axis('off')
    plt.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

plt.suptitle('SIH26142 Sentinel-2 Super-Resolution & Uncertainty', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('data/outputs/sr_comparison.png', dpi=200, bbox_inches='tight')
plt.show()


## Cell 10 — Launch Interactive Streamlit Dashboard (GPU Powered)
Runs the SRM Web App inside Colab with a secure public tunnel (`localtunnel`).

1. Run the cell below.
2. Copy the **Tunnel Password (IP)** displayed.
3. Click the **loca.lt URL**, paste the IP into the box, and click Submit!

In [ ]:
!pip install -q streamlit streamlit-folium streamlit-image-comparison pydeck plotly
!npm install -g localtunnel -q

import urllib
public_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
print('=' * 60)
print(f'YOUR TUNNEL PASSWORD (IP): {public_ip}')
print('=' * 60)

!streamlit run dashboard/app.py --server.port 8501 --server.headless true & npx localtunnel --port 8501


## Cell 11 — Download Outputs & Checkpoint
Downloads your fine-tuned model checkpoint and super-resolved GeoTIFF files.

In [ ]:
from google.colab import files
import os

to_download = [
    'src/checkpoints/model_finetuned_best.pth',
    'src/checkpoints/training_history.json',
    'src/checkpoints/training_curves.png',
    'data/outputs/sr_delhi_512_finetuned.tif',
    'data/outputs/sr_comparison.png',
]

for path in to_download:
    if os.path.exists(path):
        print(f'Downloading {path}...')
        files.download(path)
    else:
        print(f'SKIP (not found): {path}')

print('Done!')
